# P09 Privacy-Preserving Data Pipeline 源码全流程 Notebook

这本 notebook 不是通过 `subprocess` 调壳执行脚本，而是把 `project_9_privacy_pipeline` 里的完整流程源码直接展开到 notebook 中。

你可以把它当成一本按项目顺序组织的代码讲义来阅读：

- 主流程脚本按推荐执行顺序排在前面
- 辅助脚本和工具脚本排在后面
- 如果 README 里提到某个脚本但仓库里缺失，也会保留缺口说明

## 项目环境

- 项目目录：`project_9_privacy_pipeline`
- 建议 Conda 环境：`p9-privacy`

## 本 notebook 收录的源码文件顺序

1. `src/build_privacy_specs.py` - 生成隐私规格与策略 [存在]
2. `src/run_privacy_pipeline.py` - 执行隐私处理流水线 [存在]
3. `src/simulate_privacy_ops.py` - 模拟运维与事件处理 [存在]
4. `src/evaluate_privacy_pipeline.py` - 评估隐私流水线 [存在]
5. `src/run_p9_checks.py` - 项目检查 [存在]
6. `src/pipeline_utils.py` - 辅助脚本 [存在]

## 关键产物

- `data/processed/compliance_scope.json`
- `data/processed/classification_policy.json`
- `data/processed/access_policy.json`
- `data/processed/privacy_tech_options.json`
- `data/processed/raw_sensitive_records.jsonl`
- `data/processed/classified_records.jsonl`
- `data/processed/redacted_records.jsonl`
- `data/processed/quarantine_records.jsonl`
- `data/processed/audit_log.jsonl`
- `data/processed/access_alerts.jsonl`
- `data/processed/isolation_plan.json`
- `data/processed/preflight_checklist.json`
- `data/processed/incident_simulation.json`
- `data/processed/postmortem_report.json`
- `data/reports/p9_report.md`
- `data/reports/p9_metrics.json`
- `data/reports/p9_test_results.json`
- `data/reports/p9_test_report.md`


## 项目 README

下面直接附上项目自带的 `README.md`，方便在 notebook 里连同源码一起看。

# P9 Privacy-Preserving Data Pipeline

This project builds a small, reproducible privacy-preserving data processing pipeline covering compliance scope, data classification, access boundaries, de-identification, isolation, privacy technology options, preflight checks, and incident review.

## Scope

1. Scenario and compliance goals
   - Sensitive examples from healthcare support, payroll HR, and financial KYC.
   - Clear auditability, least-privilege, and privacy risk goals.
2. Data classification and access design
   - Sensitivity levels, source-type classification, and role boundaries.
   - Access controls across raw, quarantine, redacted, and audit zones.
3. De-identification, isolation, and secure processing
   - PII detection, masking, tokenization, and quarantine handling.
   - Audit log and suspicious access alerts.
4. Privacy technology integration
   - Differential privacy, TEE, FHE, and tokenization integration notes.
5. Launch checks and postmortem
   - Preflight checklist, incident simulation, and follow-up actions.
6. Extension directions
   - Extend to cross-system privacy orchestration and stronger policy controls.

## Environment

Dedicated conda environment:

```bash
conda activate p9-privacy
```

Environment files:

- `environment.yml`
- `environment.lock.yml`

Recommended creation commands:

```bash
conda env create -f environment.yml
conda env update -n p9-privacy -f environment.lock.yml --prune
```

## Recommended Run Order

```bash
python src/build_privacy_specs.py
python src/run_privacy_pipeline.py
python src/simulate_privacy_ops.py
python src/evaluate_privacy_pipeline.py
python src/run_p9_checks.py
```

## Main Outputs

- `data/processed/compliance_scope.json`
- `data/processed/classification_policy.json`
- `data/processed/access_policy.json`
- `data/processed/privacy_tech_options.json`
- `data/processed/raw_sensitive_records.jsonl`
- `data/processed/classified_records.jsonl`
- `data/processed/redacted_records.jsonl`
- `data/processed/quarantine_records.jsonl`
- `data/processed/audit_log.jsonl`
- `data/processed/access_alerts.jsonl`
- `data/processed/isolation_plan.json`
- `data/processed/preflight_checklist.json`
- `data/processed/incident_simulation.json`
- `data/processed/postmortem_report.json`
- `data/reports/p9_report.md`
- `data/reports/p9_metrics.json`
- `data/reports/p9_test_results.json`
- `data/reports/p9_test_report.md`


## 源码目录概览

当前 `src/` 中实际存在的 Python 文件共 `6` 个：

- `src/build_privacy_specs.py`
- `src/evaluate_privacy_pipeline.py`
- `src/pipeline_utils.py`
- `src/run_p9_checks.py`
- `src/run_privacy_pipeline.py`
- `src/simulate_privacy_ops.py`


## 完整源码展开


### `src/build_privacy_specs.py` - 生成隐私规格与策略

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import PROCESSED_DIR, ensure_standard_dirs, write_json

SCOPE_FILE = PROCESSED_DIR / "compliance_scope.json"
CLASSIFICATION_FILE = PROCESSED_DIR / "classification_policy.json"
ACCESS_FILE = PROCESSED_DIR / "access_policy.json"
TECH_FILE = PROCESSED_DIR / "privacy_tech_options.json"


def build_scope() -> dict:
    return {
        "pipeline_goal": "Build a reproducible privacy-preserving data processing pipeline for highly sensitive records.",
        "example_domains": ["healthcare_support", "payroll_hr", "financial_kyc"],
        "compliance_targets": ["least_privilege", "auditability", "de-identification before analytics", "incident response readiness"],
        "risk_goals": [
            "prevent direct PII leakage to analytics consumers",
            "separate raw storage from redacted processing zones",
            "log suspicious access and export attempts",
        ],
    }


def build_classification_policy() -> dict:
    return {
        "sensitivity_levels": [
            {"level": "restricted", "description": "direct PII, health identifiers, payroll details, bank data"},
            {"level": "confidential", "description": "internal case details and support notes"},
            {"level": "internal", "description": "aggregate metrics and sanitized analytics outputs"},
        ],
        "source_types": [
            {"source_type": "support_ticket", "default_level": "confidential"},
            {"source_type": "employee_payroll", "default_level": "restricted"},
            {"source_type": "kyc_form", "default_level": "restricted"},
            {"source_type": "analytics_export", "default_level": "internal"},
        ],
        "pii_rules": [
            {"pattern_name": "email", "action": "tokenize"},
            {"pattern_name": "phone", "action": "mask"},
            {"pattern_name": "ssn", "action": "remove"},
            {"pattern_name": "bank_account", "action": "tokenize"},
            {"pattern_name": "patient_id", "action": "tokenize"},
        ],
    }


def build_access_policy() -> dict:
    roles = {
        "privacy_admin": ["read_raw_zone", "approve_exports", "review_alerts", "trigger_incident_lockdown"],
        "data_processor": ["write_redacted_zone", "run_deid_jobs", "view_classification_reports"],
        "analyst": ["read_redacted_zone", "view_aggregates"],
        "auditor": ["read_audit_log", "read_alerts", "review_exception_cases"],
        "incident_responder": ["read_alerts", "lock_accounts", "quarantine_exports", "annotate_postmortem"],
    }
    return {
        "roles": roles,
        "boundary_rules": [
            "raw_zone is accessible only by privacy_admin",
            "redacted_zone is accessible by data_processor and analyst",
            "export requires privacy_admin approval and audit record",
            "suspicious cross-role access triggers alert and temporary quarantine",
        ],
        "storage_zones": [
            {"zone_name": "raw_zone", "encryption": "required", "network_scope": "isolated"},
            {"zone_name": "quarantine_zone", "encryption": "required", "network_scope": "isolated"},
            {"zone_name": "redacted_zone", "encryption": "required", "network_scope": "analytics"},
            {"zone_name": "audit_zone", "encryption": "required", "network_scope": "security"},
        ],
    }


def build_tech_options() -> list[dict]:
    return [
        {
            "technology": "differential_privacy",
            "fit": "aggregate reporting and analytics release",
            "benefit": "reduces re-identification risk for published statistics",
            "cost_tradeoff": "slight metric noise and tuning overhead",
        },
        {
            "technology": "tee",
            "fit": "isolated processing of raw restricted records",
            "benefit": "stronger runtime isolation for sensitive compute",
            "cost_tradeoff": "higher infra complexity and hardware dependency",
        },
        {
            "technology": "fhe",
            "fit": "future encrypted computation on highest-risk fields",
            "benefit": "compute on encrypted values without plaintext exposure",
            "cost_tradeoff": "significant latency and engineering cost",
        },
        {
            "technology": "tokenization",
            "fit": "operational de-identification before analytics",
            "benefit": "practical and cheap for replacing direct identifiers",
            "cost_tradeoff": "still needs secure token vault management",
        },
    ]


def main() -> None:
    ensure_standard_dirs()
    scope = build_scope()
    classification = build_classification_policy()
    access = build_access_policy()
    tech = build_tech_options()
    summary = {
        "domain_count": len(scope["example_domains"]),
        "compliance_target_count": len(scope["compliance_targets"]),
        "role_count": len(access["roles"]),
        "storage_zone_count": len(access["storage_zones"]),
        "technology_count": len(tech),
        "source_type_distribution": dict(Counter(item["default_level"] for item in classification["source_types"])),
    }
    write_json(scope, SCOPE_FILE)
    write_json(classification, CLASSIFICATION_FILE)
    write_json(access, ACCESS_FILE)
    write_json(tech, TECH_FILE)
    print("✅ P9 合规、分类、权限与隐私技术规格生成完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/run_privacy_pipeline.py` - 执行隐私处理流水线

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import re
from collections import Counter

from pipeline_utils import PROCESSED_DIR, ensure_standard_dirs, hash_token, load_json, write_json, write_jsonl

RAW_FILE = PROCESSED_DIR / "raw_sensitive_records.jsonl"
CLASSIFIED_FILE = PROCESSED_DIR / "classified_records.jsonl"
REDACTED_FILE = PROCESSED_DIR / "redacted_records.jsonl"
QUARANTINE_FILE = PROCESSED_DIR / "quarantine_records.jsonl"
AUDIT_FILE = PROCESSED_DIR / "audit_log.jsonl"
ALERTS_FILE = PROCESSED_DIR / "access_alerts.jsonl"
ISOLATION_FILE = PROCESSED_DIR / "isolation_plan.json"
SUMMARY_FILE = PROCESSED_DIR / "privacy_pipeline_summary.json"

EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"\b\d{3}-\d{3}-\d{4}\b")
SSN_RE = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
BANK_RE = re.compile(r"\b\d{10,12}\b")
PATIENT_RE = re.compile(r"\bPT-\d{4,6}\b")


def build_raw_records() -> list[dict]:
    return [
        {
            "record_id": "rec_001",
            "source_type": "support_ticket",
            "domain": "healthcare_support",
            "owner_team": "care_ops",
            "payload": "Patient John Lee, patient id PT-483920, email john.lee@example.com, phone 415-555-2198 asked about claim denial.",
        },
        {
            "record_id": "rec_002",
            "source_type": "employee_payroll",
            "domain": "payroll_hr",
            "owner_team": "hr_ops",
            "payload": "Employee Marta Chen SSN 342-19-8842 salary adjustment note for payroll cycle 2026-04.",
        },
        {
            "record_id": "rec_003",
            "source_type": "kyc_form",
            "domain": "financial_kyc",
            "owner_team": "fin_ops",
            "payload": "KYC form for beta@corp.test with bank account 998877665544 and risk review pending.",
        },
        {
            "record_id": "rec_004",
            "source_type": "support_ticket",
            "domain": "healthcare_support",
            "owner_team": "care_ops",
            "payload": "Follow-up ticket for patient PT-555888, callback 212-555-0100, no diagnosis shared.",
        },
        {
            "record_id": "rec_005",
            "source_type": "analytics_export",
            "domain": "financial_kyc",
            "owner_team": "analytics",
            "payload": "Weekly aggregate: 18 forms reviewed, 2 pending, 0 escalated.",
        },
        {
            "record_id": "rec_006",
            "source_type": "employee_payroll",
            "domain": "payroll_hr",
            "owner_team": "hr_ops",
            "payload": "Payroll support case for sam.green@example.com direct deposit account 123456789012 requires verification.",
        },
        {
            "record_id": "rec_007",
            "source_type": "kyc_form",
            "domain": "financial_kyc",
            "owner_team": "fin_ops",
            "payload": "Applicant id review for arya@startup.test, analyst requested additional address proof.",
        },
        {
            "record_id": "rec_008",
            "source_type": "support_ticket",
            "domain": "healthcare_support",
            "owner_team": "care_ops",
            "payload": "Ticket notes mention guardian contact at 650-555-8821 and email family.contact@test.org.",
        },
    ]


def detect_pii(text: str) -> list[dict]:
    detections: list[dict] = []
    for pattern_name, regex in [
        ("email", EMAIL_RE),
        ("phone", PHONE_RE),
        ("ssn", SSN_RE),
        ("bank_account", BANK_RE),
        ("patient_id", PATIENT_RE),
    ]:
        for match in regex.finditer(text):
            detections.append({"pattern_name": pattern_name, "match": match.group(0)})
    return detections


def redact_payload(text: str, detections: list[dict]) -> str:
    redacted = text
    for detection in detections:
        match = detection["match"]
        if detection["pattern_name"] in {"email", "bank_account", "patient_id"}:
            replacement = hash_token(match)
        elif detection["pattern_name"] == "phone":
            replacement = "***-***-" + match[-4:]
        else:
            replacement = "[REMOVED_SSN]"
        redacted = redacted.replace(match, replacement)
    return redacted


def classify_record(record: dict, classification_policy: dict) -> dict:
    source_type_map = {item["source_type"]: item["default_level"] for item in classification_policy["source_types"]}
    detections = detect_pii(record["payload"])
    sensitivity = source_type_map.get(record["source_type"], "internal")
    if detections:
        sensitivity = "restricted"
    return {
        **record,
        "sensitivity_level": sensitivity,
        "pii_detections": detections,
        "requires_quarantine": sensitivity == "restricted",
    }


def build_audit_log() -> list[dict]:
    return [
        {"event_id": "audit_001", "actor": "privacy_admin", "action": "ingest_raw_batch", "target": "raw_zone"},
        {"event_id": "audit_002", "actor": "data_processor", "action": "run_deid_job", "target": "redacted_zone"},
        {"event_id": "audit_003", "actor": "analyst", "action": "read_redacted_export", "target": "redacted_zone"},
        {"event_id": "audit_004", "actor": "analyst", "action": "attempt_read_raw_zone", "target": "raw_zone"},
        {"event_id": "audit_005", "actor": "incident_responder", "action": "quarantine_export", "target": "quarantine_zone"},
    ]


def build_alerts() -> list[dict]:
    return [
        {
            "alert_id": "alert_priv_001",
            "severity": "high",
            "actor": "analyst",
            "reason": "unauthorized raw zone access attempt",
            "status": "resolved",
        },
        {
            "alert_id": "alert_priv_002",
            "severity": "medium",
            "actor": "data_processor",
            "reason": "restricted export requested without approval",
            "status": "resolved",
        },
    ]


def build_isolation_plan() -> dict:
    return {
        "zones": [
            {"zone_name": "raw_zone", "store": "encrypted object storage", "access": ["privacy_admin"]},
            {"zone_name": "quarantine_zone", "store": "isolated secure bucket", "access": ["privacy_admin", "incident_responder"]},
            {"zone_name": "redacted_zone", "store": "analytics warehouse", "access": ["data_processor", "analyst"]},
            {"zone_name": "audit_zone", "store": "security log store", "access": ["auditor", "privacy_admin"]},
        ],
        "deid_flow": [
            "ingest raw restricted records",
            "classify and detect PII",
            "write restricted originals to raw_zone",
            "redact identifiers and emit sanitized records to redacted_zone",
            "quarantine flagged export attempts and emit audit alerts",
        ],
    }


def main() -> None:
    ensure_standard_dirs()
    classification_policy = load_json(PROCESSED_DIR / "classification_policy.json")
    raw_records = build_raw_records()
    classified = [classify_record(record, classification_policy) for record in raw_records]
    redacted = []
    quarantined = []
    for record in classified:
        redacted_record = dict(record)
        redacted_record["payload"] = redact_payload(record["payload"], record["pii_detections"])
        redacted.append(redacted_record)
        if record["requires_quarantine"]:
            quarantined.append({"record_id": record["record_id"], "reason": "restricted_source", "owner_team": record["owner_team"]})

    audit_log = build_audit_log()
    alerts = build_alerts()
    isolation_plan = build_isolation_plan()
    summary = {
        "raw_record_count": len(raw_records),
        "classified_record_count": len(classified),
        "restricted_record_count": sum(item["sensitivity_level"] == "restricted" for item in classified),
        "redacted_record_count": len(redacted),
        "quarantine_count": len(quarantined),
        "pii_detection_distribution": dict(Counter(detection["pattern_name"] for item in classified for detection in item["pii_detections"])),
        "alert_count": len(alerts),
        "audit_event_count": len(audit_log),
    }

    write_jsonl(raw_records, RAW_FILE)
    write_jsonl(classified, CLASSIFIED_FILE)
    write_jsonl(redacted, REDACTED_FILE)
    write_jsonl(quarantined, QUARANTINE_FILE)
    write_jsonl(audit_log, AUDIT_FILE)
    write_jsonl(alerts, ALERTS_FILE)
    write_json(isolation_plan, ISOLATION_FILE)
    write_json(summary, SUMMARY_FILE)
    print("✅ P9 脱敏、隔离与安全处理流水线完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/simulate_privacy_ops.py` - 模拟运维与事件处理

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from pipeline_utils import PROCESSED_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

PREFLIGHT_FILE = PROCESSED_DIR / "preflight_checklist.json"
INCIDENT_FILE = PROCESSED_DIR / "incident_simulation.json"
POSTMORTEM_FILE = PROCESSED_DIR / "postmortem_report.json"
OPS_SUMMARY_FILE = PROCESSED_DIR / "ops_summary.json"


def main() -> None:
    ensure_standard_dirs()
    classified = load_jsonl(PROCESSED_DIR / "classified_records.jsonl")
    alerts = load_jsonl(PROCESSED_DIR / "access_alerts.jsonl")
    access_policy = load_json(PROCESSED_DIR / "access_policy.json")
    tech_options = load_json(PROCESSED_DIR / "privacy_tech_options.json")

    preflight = {
        "checks": [
            {"name": "all records classified", "passed": len(classified) > 0 and all("sensitivity_level" in item for item in classified)},
            {"name": "restricted records isolated", "passed": all(item["requires_quarantine"] == (item["sensitivity_level"] == "restricted") for item in classified)},
            {"name": "alerts wired", "passed": len(alerts) >= 2},
            {"name": "role model present", "passed": len(access_policy["roles"]) >= 5},
            {"name": "privacy tech options documented", "passed": len(tech_options) >= 4},
        ]
    }
    preflight["passed_checks"] = sum(item["passed"] for item in preflight["checks"])
    preflight["total_checks"] = len(preflight["checks"])
    preflight["overall_passed"] = preflight["passed_checks"] == preflight["total_checks"]

    incident = {
        "incident_id": "privacy_inc_001",
        "scenario": "analyst attempted to export restricted raw records without approval",
        "detection": "access_alerts.jsonl high severity alert",
        "containment": [
            "quarantine the export request",
            "lock the analyst session",
            "require privacy_admin review",
        ],
        "outcome": "resolved with no confirmed external data leak",
        "response_minutes": 24,
    }

    postmortem = {
        "incident_id": incident["incident_id"],
        "root_cause": "role boundary was respected, but analyst attempted an out-of-policy access path",
        "what_worked": [
            "alert fired immediately",
            "quarantine workflow prevented raw export",
            "audit log preserved actor timeline",
        ],
        "follow_ups": [
            "add stronger export pre-check banner for analysts",
            "require dual approval for restricted export exceptions",
            "add DP gate for aggregate release workflows",
        ],
    }

    ops_summary = {
        "preflight_pass_rate": round(preflight["passed_checks"] / max(1, preflight["total_checks"]), 4),
        "incident_response_minutes": incident["response_minutes"],
        "follow_up_count": len(postmortem["follow_ups"]),
    }

    write_json(preflight, PREFLIGHT_FILE)
    write_json(incident, INCIDENT_FILE)
    write_json(postmortem, POSTMORTEM_FILE)
    write_json(ops_summary, OPS_SUMMARY_FILE)
    print("✅ P9 上线检查与事故复盘模拟完成。")
    print(ops_summary)


if __name__ == "__main__":
    main()


### `src/evaluate_privacy_pipeline.py` - 评估隐私流水线

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import re
from collections import Counter

from pipeline_utils import PROCESSED_DIR, REPORTS_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

METRICS_FILE = REPORTS_DIR / "p9_metrics.json"
REPORT_FILE = REPORTS_DIR / "p9_report.md"

EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"\b\d{3}-\d{3}-\d{4}\b")
SSN_RE = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
BANK_RE = re.compile(r"\b\d{10,12}\b")
PATIENT_RE = re.compile(r"\bPT-\d{4,6}\b")


def has_direct_pii(text: str) -> bool:
    return any(regex.search(text) for regex in [EMAIL_RE, PHONE_RE, SSN_RE, BANK_RE, PATIENT_RE])


def main() -> None:
    ensure_standard_dirs()
    scope = load_json(PROCESSED_DIR / "compliance_scope.json")
    classification = load_json(PROCESSED_DIR / "classification_policy.json")
    access = load_json(PROCESSED_DIR / "access_policy.json")
    tech_options = load_json(PROCESSED_DIR / "privacy_tech_options.json")
    raw_records = load_jsonl(PROCESSED_DIR / "raw_sensitive_records.jsonl")
    classified = load_jsonl(PROCESSED_DIR / "classified_records.jsonl")
    redacted = load_jsonl(PROCESSED_DIR / "redacted_records.jsonl")
    quarantined = load_jsonl(PROCESSED_DIR / "quarantine_records.jsonl")
    alerts = load_jsonl(PROCESSED_DIR / "access_alerts.jsonl")
    audit_log = load_jsonl(PROCESSED_DIR / "audit_log.jsonl")
    preflight = load_json(PROCESSED_DIR / "preflight_checklist.json")
    incident = load_json(PROCESSED_DIR / "incident_simulation.json")
    postmortem = load_json(PROCESSED_DIR / "postmortem_report.json")

    direct_pii_removed_rate = round(
        sum(not has_direct_pii(item["payload"]) for item in redacted) / max(1, len(redacted)),
        4,
    )
    metrics = {
        "domain_count": len(scope["example_domains"]),
        "compliance_target_count": len(scope["compliance_targets"]),
        "source_type_count": len(classification["source_types"]),
        "role_count": len(access["roles"]),
        "privacy_tech_count": len(tech_options),
        "raw_record_count": len(raw_records),
        "restricted_record_count": sum(item["sensitivity_level"] == "restricted" for item in classified),
        "quarantine_count": len(quarantined),
        "pii_detection_distribution": dict(Counter(detection["pattern_name"] for item in classified for detection in item["pii_detections"])),
        "direct_pii_removed_rate": direct_pii_removed_rate,
        "alert_count": len(alerts),
        "resolved_alert_rate": round(sum(item["status"] == "resolved" for item in alerts) / max(1, len(alerts)), 4),
        "audit_event_count": len(audit_log),
        "preflight_pass_rate": round(preflight["passed_checks"] / max(1, preflight["total_checks"]), 4),
        "incident_response_minutes": incident["response_minutes"],
        "postmortem_follow_up_count": len(postmortem["follow_ups"]),
    }
    write_json(metrics, METRICS_FILE)

    report = f"""# P9 Privacy-Preserving Data Pipeline Report

## 1. 场景与合规目标

- 场景域数：{metrics['domain_count']}
- 合规目标数：{metrics['compliance_target_count']}
- 数据源类型数：{metrics['source_type_count']}
- 角色数：{metrics['role_count']}

## 2. 数据分类与权限设计

- 原始记录数：{metrics['raw_record_count']}
- Restricted 记录数：{metrics['restricted_record_count']}
- 隔离/隔离待审记录数：{metrics['quarantine_count']}

## 3. 脱敏、隔离与安全处理

- PII 检测分布：{metrics['pii_detection_distribution']}
- 直接 PII 去除率：{metrics['direct_pii_removed_rate']}
- 告警数：{metrics['alert_count']}
- 审计事件数：{metrics['audit_event_count']}

## 4. 隐私保护技术接入

- 隐私技术选项数：{metrics['privacy_tech_count']}
- 当前演示覆盖差分隐私、TEE、FHE 与 tokenization 接入说明。

## 5. 上线检查与复盘

- 预上线检查通过率：{metrics['preflight_pass_rate']}
- 事故响应时长：{metrics['incident_response_minutes']} 分钟
- 复盘后续动作数：{metrics['postmortem_follow_up_count']}

## 6. 扩展方向

- 从单条流水线扩展到跨系统隐私编排与跨组织协作。
- 引入更细粒度 purpose-based access control 和数据保留策略。
- 将隐私预算与导出审批打通到更完整的数据平台控制面。
"""
    REPORT_FILE.write_text(report, encoding="utf-8")
    print("✅ P9 报告生成完成。")
    print(report)


if __name__ == "__main__":
    main()


### `src/run_p9_checks.py` - 项目检查

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import re
import subprocess
from pathlib import Path

from pipeline_utils import PROCESSED_DIR, REPORTS_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

ROOT_DIR = Path(__file__).resolve().parent.parent
SRC_DIR = ROOT_DIR / "src"
RESULTS_FILE = REPORTS_DIR / "p9_test_results.json"
REPORT_FILE = REPORTS_DIR / "p9_test_report.md"

EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"\b\d{3}-\d{3}-\d{4}\b")
SSN_RE = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
BANK_RE = re.compile(r"\b\d{10,12}\b")
PATIENT_RE = re.compile(r"\bPT-\d{4,6}\b")


def run_command(command: list[str], name: str) -> dict:
    result = subprocess.run(command, capture_output=True, text=True)
    return {
        "name": name,
        "command": command,
        "returncode": result.returncode,
        "passed": result.returncode == 0,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
    }


def has_direct_pii(text: str) -> bool:
    return any(regex.search(text) for regex in [EMAIL_RE, PHONE_RE, SSN_RE, BANK_RE, PATIENT_RE])


def main() -> None:
    ensure_standard_dirs()
    scope = load_json(PROCESSED_DIR / "compliance_scope.json")
    classification = load_json(PROCESSED_DIR / "classification_policy.json")
    access = load_json(PROCESSED_DIR / "access_policy.json")
    tech = load_json(PROCESSED_DIR / "privacy_tech_options.json")
    raw_records = load_jsonl(PROCESSED_DIR / "raw_sensitive_records.jsonl")
    classified = load_jsonl(PROCESSED_DIR / "classified_records.jsonl")
    redacted = load_jsonl(PROCESSED_DIR / "redacted_records.jsonl")
    quarantined = load_jsonl(PROCESSED_DIR / "quarantine_records.jsonl")
    alerts = load_jsonl(PROCESSED_DIR / "access_alerts.jsonl")
    audit_log = load_jsonl(PROCESSED_DIR / "audit_log.jsonl")
    preflight = load_json(PROCESSED_DIR / "preflight_checklist.json")
    incident = load_json(PROCESSED_DIR / "incident_simulation.json")
    postmortem = load_json(PROCESSED_DIR / "postmortem_report.json")

    py_files = sorted(str(path) for path in SRC_DIR.glob("*.py"))
    command_checks = [
        run_command(["python", "-m", "py_compile", *py_files], "py_compile"),
        run_command(["python", str(SRC_DIR / "evaluate_privacy_pipeline.py")], "evaluate_privacy_pipeline"),
    ]

    dataset_checks = []
    dataset_checks.append(
        {
            "name": "required_files_exist",
            "passed": all(
                path.exists()
                for path in [
                    PROCESSED_DIR / "compliance_scope.json",
                    PROCESSED_DIR / "classification_policy.json",
                    PROCESSED_DIR / "access_policy.json",
                    PROCESSED_DIR / "privacy_tech_options.json",
                    PROCESSED_DIR / "raw_sensitive_records.jsonl",
                    PROCESSED_DIR / "classified_records.jsonl",
                    PROCESSED_DIR / "redacted_records.jsonl",
                    PROCESSED_DIR / "quarantine_records.jsonl",
                    PROCESSED_DIR / "audit_log.jsonl",
                    PROCESSED_DIR / "access_alerts.jsonl",
                    PROCESSED_DIR / "preflight_checklist.json",
                    PROCESSED_DIR / "incident_simulation.json",
                    PROCESSED_DIR / "postmortem_report.json",
                ]
            ),
            "details": {},
        }
    )
    dataset_checks.append(
        {
            "name": "role_and_zone_model_present",
            "passed": len(access["roles"]) >= 5 and len(access["storage_zones"]) >= 4,
            "details": {"role_count": len(access["roles"]), "storage_zone_count": len(access["storage_zones"])},
        }
    )
    dataset_checks.append(
        {
            "name": "all_records_classified",
            "passed": len(raw_records) == len(classified) and all("sensitivity_level" in item for item in classified),
            "details": {"raw_record_count": len(raw_records), "classified_record_count": len(classified)},
        }
    )
    dataset_checks.append(
        {
            "name": "restricted_records_quarantined",
            "passed": sum(item["sensitivity_level"] == "restricted" for item in classified) == len(quarantined),
            "details": {"restricted_count": sum(item["sensitivity_level"] == "restricted" for item in classified), "quarantine_count": len(quarantined)},
        }
    )
    dataset_checks.append(
        {
            "name": "redacted_records_remove_direct_pii",
            "passed": all(not has_direct_pii(item["payload"]) for item in redacted),
            "details": {"redacted_record_count": len(redacted)},
        }
    )
    dataset_checks.append(
        {
            "name": "pii_detection_rules_present",
            "passed": {"email", "phone", "ssn", "bank_account", "patient_id"} == {item["pattern_name"] for item in classification["pii_rules"]},
            "details": {"pii_rule_names": [item["pattern_name"] for item in classification["pii_rules"]]},
        }
    )
    dataset_checks.append(
        {
            "name": "alerts_and_audit_present",
            "passed": len(alerts) >= 2 and len(audit_log) >= 5,
            "details": {"alert_count": len(alerts), "audit_event_count": len(audit_log)},
        }
    )
    dataset_checks.append(
        {
            "name": "alerts_resolved",
            "passed": all(item["status"] == "resolved" for item in alerts),
            "details": {"resolved_alerts": sum(item["status"] == "resolved" for item in alerts)},
        }
    )
    dataset_checks.append(
        {
            "name": "privacy_tech_options_complete",
            "passed": {"differential_privacy", "tee", "fhe", "tokenization"} == {item["technology"] for item in tech},
            "details": {"technology_names": [item["technology"] for item in tech]},
        }
    )
    dataset_checks.append(
        {
            "name": "preflight_all_passed",
            "passed": preflight["overall_passed"],
            "details": {"passed_checks": preflight["passed_checks"], "total_checks": preflight["total_checks"]},
        }
    )
    dataset_checks.append(
        {
            "name": "incident_and_postmortem_present",
            "passed": incident["response_minutes"] <= 30 and len(postmortem["follow_ups"]) >= 3,
            "details": {"response_minutes": incident["response_minutes"], "follow_up_count": len(postmortem["follow_ups"])},
        }
    )

    overall_passed = all(item["passed"] for item in command_checks + dataset_checks)
    results = {
        "overall_passed": overall_passed,
        "total_checks": len(command_checks) + len(dataset_checks),
        "passed_checks": sum(item["passed"] for item in command_checks + dataset_checks),
        "command_checks": command_checks,
        "dataset_checks": dataset_checks,
    }
    write_json(results, RESULTS_FILE)

    lines = [
        "# P9 Test Report",
        "",
        f"- Overall status: {'PASS' if overall_passed else 'FAIL'}",
        f"- Passed checks: {results['passed_checks']}/{results['total_checks']}",
        "",
        "## Command Checks",
        "",
    ]
    for item in command_checks:
        lines.append(f"- {item['name']}: {'PASS' if item['passed'] else 'FAIL'}")
        lines.append(f"  - Command: `{' '.join(item['command'])}`")
    lines.extend(["", "## Dataset Checks", ""])
    for item in dataset_checks:
        lines.append(f"- {item['name']}: {'PASS' if item['passed'] else 'FAIL'}")
        lines.append(f"  - Details: `{item['details']}`")

    REPORT_FILE.write_text("\n".join(lines), encoding="utf-8")
    print("✅ P9 测试完成。")
    print(results)


if __name__ == "__main__":
    main()


### `src/pipeline_utils.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Iterable

ROOT_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = DATA_DIR / "reports"


def ensure_standard_dirs() -> None:
    for path in [DATA_DIR, PROCESSED_DIR, REPORTS_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def write_json(data: dict | list, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def load_json(path: Path) -> dict | list:
    return json.loads(path.read_text(encoding="utf-8"))


def write_jsonl(records: Iterable[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    records: list[dict] = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def hash_token(value: str) -> str:
    digest = hashlib.sha256(value.encode("utf-8")).hexdigest()
    return f"tok_{digest[:12]}"
